## ETL FDP - 2024

# Before
### Screenshot of the file before running the ETL process.
### The payroll spreadsheet generated by the accounting system was poorly structured and unformatted, making data analysis difficult and impractical.
### In addition, consolidating the large volume of data required significant manual effort from the department, often taking several days to complete the process.

![before](https://raw.githubusercontent.com/Magno-Rodrigues/etl-fdp/main/notebooks/images/before.png)


In [ ]:
# Excel ETL Processing with Pandas (Highly Documented Version)

# ============================================================
# IMPORTING LIBRARIES
# ============================================================

# Pandas:
# Main library used for tabular data manipulation.
#
# Used for:
# - reading Excel files
# - DataFrame manipulation
# - ETL processing
# - Excel export
import pandas as pd


# ============================================================
# LOADING THE EXCEL FILE
# ============================================================

# read_excel():
# Reads the .xls file.
#
# engine='xlrd':
# Required for old Excel file formats (.xls)
df_csv = pd.read_excel(
    r"PATH\Extrato Mensal Consulting - 012024 (POR TOMADOR).xls",
    engine='xlrd'
)


# ============================================================
# LIST OF CODES THAT WILL BECOME COLUMNS
# ============================================================

# Each value in this list will become
# a column in the final DataFrame.
#
# Example:
# code 1  -> column "1"
# code 10 -> column "10"
# code 19 -> column "19"
novas_colunas = [

    1.0, 10.0, 19.0, 25.0, 22.0, 28.0,
    29.0, 64.0, 149.0, 150.0, 160.0,

    200.0, 235.0, 236.0, 245.0, 246.0,
    250.0, 270.0, 271.0, 272.0, 275.0,

    276.0, 277.0, 278.0, 279.0, 282.0,
    283.0, 289.0, 292.0, 293.0, 301.0,

    302.0, 303.0, 307.0, 325.0, 326.0,
    327.0, 328.0, 330.0, 331.0, 335.0,

    336.0, 337.0, 338.0, 339.0, 340.0,
    341.0, 342.0, 343.0, 344.0, 345.0,

    346.0, 347.0, 348.0, 350.0, 351.0,
    352.0, 372.0, 374.0, 375.0, 383.0,

    392.0, 393.0, 802.0, 803.0, 805.0,
    806.0, 807.0, 808.0, 809.0, 810.0,

    811.0, 815.0, 816.0, 817.0, 818.0,
    819.0, 820.0, 831.0, 832.0, 836.0,

    846.0, 854.0, 931.0, 932.0, 940.0,
    990.0, 8081.0, 8082.0, 8083.0, 8085.0,

    8087.0, 8088.0, 8112.0, 8126.0, 8130.0,
    8139.0, 8144.0, 8145.0, 8146.0, 8152.0,

    8153.0, 8154.0, 8158.0, 8156.0, 8157.0,
    8169.0, 8181.0, 8182.0, 8184.0, 8185.0,

    8186.0, 8189.0, 8190.0, 8192.0, 8193.0,
    8194.0, 8197.0, 8198.0, 8201.0, 8200.0,

    8202.0, 8294.0, 8296.0, 8299.0, 8307.0,
    8309.0, 8312.0, 8314.0, 8316.0, 8324.0,

    8326.0, 8328.0, 8374.0, 8375.0, 8378.0,
    8379.0, 8381.0, 8392.0, 8393.0, 8394.0,

    8399.0, 8401.0, 8417.0, 8427.0, 8467.0,
    8489.0, 8546.0, 8550.0, 8551.0, 8552.0,

    8553.0, 8556.0, 8781.0, 8783.0, 8784.0,
    8785.0, 8791.0, 8797.0, 8800.0, 8831.0,

    8832.0, 8869.0, 8870.0, 8917.0, 9180.0,
    9235.0, 9360.0, 9361.0, 9365.0, 9380.0,

    9403.0, 9489.0, 9522.0, 9591.0, 9592.0,
    9594.0, 9595.0, 9596.0, 9597.0, 9600.0,
    9601.0
]


# ============================================================
# CONVERTING CODES TO INTEGER
# ============================================================

# Removes ".0"
#
# Example:
# 10.0 -> 10
# 25.0 -> 25
novas_colunas = [

    int(col)

    for col in novas_colunas
]


# ============================================================
# CREATING THE OUTPUT DATAFRAME
# ============================================================

# The final DataFrame will contain:
#
# Fixed columns:
# - CÓDIGO
# - NOME
# - SITUAÇÃO
# - CPF
#
# Plus all dynamic columns from the list.
df_resultado = pd.DataFrame(

    columns=[

        'CÓDIGO',
        'NOME',
        'SITUAÇÃO',
        'CPF'

    ] + [str(col) for col in novas_colunas]
)


# ============================================================
# FUNCTION RESPONSIBLE FOR BUILDING EACH ROW
# ============================================================

def preencher_linha(cargo_index):

    # --------------------------------------------------------
    # INDEX ADJUSTMENT
    # --------------------------------------------------------

    # The required information is located
    # two rows above the line containing "Cargo:"
    ajuste_index = cargo_index - 2

    # Prevents negative indexes
    if ajuste_index < 0:

        return pd.Series(dtype=object)


    # --------------------------------------------------------
    # FIXED DATA
    # --------------------------------------------------------

    # Extracts:
    # - code
    # - name
    # - status
    # - CPF
    #
    # These values come from the adjusted row.
    nova_linha = pd.Series(

        df_csv.loc[
            ajuste_index,
            [
                'Unnamed: 5',
                'Unnamed: 10',
                'Unnamed: 36',
                'Unnamed: 55'
            ]
        ].values,

        index=[
            'CÓDIGO',
            'NOME',
            'SITUAÇÃO',
            'CPF'
        ]
    )


    # --------------------------------------------------------
    # INITIALIZES ALL COLUMNS AS NULL
    # --------------------------------------------------------

    for col in novas_colunas:

        nova_linha[str(col)] = pd.NA


    # --------------------------------------------------------
    # SUB-DATAFRAME
    # --------------------------------------------------------

    # Selects all rows below "Cargo:"
    sub_df = df_csv.iloc[cargo_index + 1:, :]


    # --------------------------------------------------------
    # ITERATING THROUGH ROWS BELOW "Cargo:"
    # --------------------------------------------------------

    for _, row in sub_df.iterrows():

        # Value found in the code column
        valor_na_coluna = row['Unnamed: 1']


        # ----------------------------------------------------
        # CHECKS IF THE CODE IS VALID
        # ----------------------------------------------------

        if (

            pd.notna(valor_na_coluna)

            and

            valor_na_coluna in novas_colunas
        ):

            # Fills the corresponding column
            #
            # Example:
            # code 10 -> column "10"
            nova_linha[str(int(valor_na_coluna))] = row['Unnamed: 32']


        else:

            # ------------------------------------------------
            # END OF THE SEQUENCE
            # ------------------------------------------------

            # When an invalid row is found,
            # stops reading the sequence.
            break


    # Returns the fully populated row
    return nova_linha


# ============================================================
# LIST USED TO STORE PROCESSED ROWS
# ============================================================

linhas_preenchidas = []


# ============================================================
# ITERATING THROUGH THE ENTIRE SPREADSHEET
# ============================================================

for index, row in df_csv.iterrows():

    # Searches for rows containing "Cargo:"
    if 'Cargo:' in str(row['Empresa:']):

        # Processes the row
        linha_preenchida = preencher_linha(index)

        # Checks if the row is valid
        if not linha_preenchida.empty:

            linhas_preenchidas.append(
                linha_preenchida
            )


# ============================================================
# CONVERTS FINAL RESULT INTO DATAFRAME
# ============================================================

df_resultado = pd.DataFrame(
    linhas_preenchidas
)


# ============================================================
# EXPORTING TO EXCEL
# ============================================================

# to_excel():
# Exports the processed DataFrame to Excel.
#
# engine='openpyxl':
# engine responsible for writing .xlsx files
#
# float_format:
# formats decimal numbers.
df_resultado.to_excel(

    "2024-1-CONSULTING.xlsx",

    engine='openpyxl',

    float_format="%.2f"
)


# ============================================================
# FINAL OUTPUT
# ============================================================

# Displays the final result
df_resultado

# After

### Screenshot of the file after running the ETL process.
### After the ETL process, employee data from each contract was organized according to monthly payroll earnings,
### making identification, tracking, and contract measurement validation significantly easier.
### The previously time-consuming process, which required substantial manual effort from the team, was reduced to just a few minutes.

![after](images/after.png)